<a href="https://colab.research.google.com/github/ClickDgo/DefuncionesPorDiabetesEnDurango/blob/main/Defunciones_por_Diabetes_en_Durango.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import scipy.stats as stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from scipy.stats import levene, shapiro, t

# ==========================================
# 0. CARGA Y PREPARACIÓN DE DATOS
# ==========================================
# Reemplaza 'tu_dataset.csv' con tu archivo real en Google Colab
# df = pd.read_csv('tu_dataset.csv')

# Asegúrate de nombrar las columnas exactamente así (o ajústalas):
# df.rename(columns={
#     'edad': 'Edad',
#     'sexo': 'Sexo',
#     'municipio': 'Municipio',
#     'escolaridad': 'Escolaridad',
#     'estado_civil': 'EstadoCivil'
# }, inplace=True)

print(f"Total de registros analizados (N): {len(df)}")


# ==========================================
# 1. ANÁLISIS DESCRIPTIVO JERÁRQUICO
# Orden: Sexo -> Municipio -> Escolaridad -> Estado Civil
# ==========================================
print("\n" + "="*80)
print("1. ANÁLISIS DESCRIPTIVO EN ORDEN JERÁRQUICO")
print("="*80)

print("\n--- A. Descriptivos globales de Edad por Sexo ---")
print(df.groupby('Sexo')['Edad'].agg(['count', 'mean', 'std', 'min', 'max']))

print("\n--- B. Descriptivos por Sexo y Municipio ---")
print(df.groupby(['Sexo', 'Municipio'])['Edad'].agg(['count', 'mean', 'std']))

print("\n--- C. Descriptivos por Sexo, Municipio y Escolaridad ---")
print(df.groupby(['Sexo', 'Municipio', 'Escolaridad'])['Edad'].agg(['count', 'mean']))

print("\n--- D. Descriptivos por Sexo, Municipio, Escolaridad y Estado Civil ---")
print(df.groupby(['Sexo', 'Municipio', 'Escolaridad', 'EstadoCivil'])['Edad'].agg(['count', 'mean']))


# ==========================================
# 2. PRUEBAS DE SUPUESTOS DE RESIDUOS
# ==========================================
print("\n" + "="*80)
print("2. VALIDACIÓN DE SUPUESTOS (NORMALIDAD Y HOMOCEDASTICIDAD)")
print("="*80)

# Modelo factorial completo con interacciones clave (ej. Sexo:Escolaridad, Municipio:Escolaridad)
formula_modelo = ('Edad ~ C(Sexo) + C(Municipio) + C(Escolaridad) + C(EstadoCivil) + '
                  'C(Sexo):C(Escolaridad) + C(Municipio):C(Escolaridad)')

model_ols = smf.ols(formula_modelo, data=df).fit()
residuos = model_ols.resid

# Shapiro-Wilk para normalidad de residuos
stat_sw, p_sw = shapiro(residuos)
print(f"Shapiro-Wilk (Normalidad de residuos) -> p-valor: {p_sw:.5f} ({'Cumple' if p_sw > 0.05 else 'No cumple'})")

# Levene para homocedasticidad (agrupado por Escolaridad como factor principal)
groups_lev = [group['Edad'].values for name, group in df.groupby('Escolaridad')]
stat_lev, p_lev = levene(*groups_lev)
print(f"Levene (Homocedasticidad) -> p-valor: {p_lev:.5f} ({'Cumple' if p_lev > 0.05 else 'No cumple'})")


# ==========================================
# 3. ANOVA FACTORIAL CON ERRORES ROBUSTOS (HC3)
# ==========================================
print("\n" + "="*80)
print("3. ANOVA FACTORIAL CON CORRECCIÓN ROBUSTA HC3")
print("="*80)

# ANOVA Tipo 2 con covarianza robusta HC3 (ideal ante desviaciones de supuestos)
anova_hc3 = sm.stats.anova_lm(model_ols, typ=2, robust='HC3')
print(anova_hc3)


# ==========================================
# 4. PRUEBAS POST-HOC (TUKEY HSD Y DUNCAN)
# ==========================================
print("\n" + "="*80)
print("4. PRUEBAS POST-HOC (FACTORES CON > 2 NIVELES)")
print("="*80)

from scipy.stats import studentized_range

# Función auxiliar corregida para la Prueba de Rangos Múltiples de Duncan
def duncan_test(data, response_col, factor_col, alpha=0.05):
    df_mean = data.groupby(factor_col)[response_col].agg(['mean', 'count', 'std']).reset_index()
    df_mean = df_mean.sort_values(by='mean', ascending=True).reset_index(drop=True)

    mse = model_ols.scale
    df_err = model_ols.df_resid
    k = len(df_mean)

    results = []
    for i in range(k):
        for j in range(i + 1, k):
            g1 = df_mean.loc[i]
            g2 = df_mean.loc[j]
            diff = abs(g1['mean'] - g2['mean'])
            p = j - i + 1  # Número de pasos entre medias ordenadas

            # Valor crítico del rango estudentizado para Duncan
            q_val = studentized_range.ppf(1.0 - alpha**(p-1), p, df_err)
            lsr = q_val * np.sqrt(mse / g1['count'])

            sig = "Significativo" if diff > lsr else "No significativo"
            results.append({
                'Grupo 1': g1[factor_col],
                'Grupo 2': g2[factor_col],
                'Dif. Medias': round(diff, 4),
                'LSR': round(lsr, 4),
                'Resultado': sig
            })
    return pd.DataFrame(results)

# A. Para Escolaridad (> 2 niveles)
print("\n--- Tukey HSD: Escolaridad ---")
print(pairwise_tukeyhsd(endog=df['Edad'], groups=df['Escolaridad'], alpha=0.05))

print("\n--- Duncan Test: Escolaridad ---")
print(duncan_test(df, 'Edad', 'Escolaridad'))

# B. Para Municipio (> 2 niveles)
if df['Municipio'].nunique() > 2:
    print("\n--- Tukey HSD: Municipio ---")
    print(pairwise_tukeyhsd(endog=df['Edad'], groups=df['Municipio'], alpha=0.05))

    print("\n--- Duncan Test: Municipio ---")
    print(duncan_test(df, 'Edad', 'Municipio'))

# C. Para Estado Civil (> 2 niveles)
if df['EstadoCivil'].nunique() > 2:
    print("\n--- Tukey HSD: Estado Civil ---")
    print(pairwise_tukeyhsd(endog=df['Edad'], groups=df['EstadoCivil'], alpha=0.05))

    print("\n--- Duncan Test: Estado Civil ---")
    print(duncan_test(df, 'Edad', 'EstadoCivil'))

Total de registros analizados (N): 466

1. ANÁLISIS DESCRIPTIVO EN ORDEN JERÁRQUICO

--- A. Descriptivos globales de Edad por Sexo ---
        count       mean        std  min  max
Sexo                                         
Hombre    236  69.504237  12.861123   27   96
Mujer     230  73.630435  13.051056   22   99

--- B. Descriptivos por Sexo y Municipio ---
                      count       mean        std
Sexo   Municipio                                 
Hombre Durango           88  68.920455  12.943930
       Gomez Palacio    112  69.187500  12.878875
       Lerdo             36  71.916667  12.686044
Mujer  Durango          111  74.873874  12.037152
       Gomez Palacio     88  71.784091  14.676159
       Lerdo             31  74.419355  11.310096

--- C. Descriptivos por Sexo, Municipio y Escolaridad ---
                                      count       mean
Sexo   Municipio     Escolaridad                      
Hombre Durango       Bachillerato         4  66.250000
           

In [ ]:
# 1. Generar el código HTML de las gráficas (usando CDN para que cargue la librería Plotly)
html_esc = fig_esc.to_html(full_html=False, include_plotlyjs='cdn')
html_ec = fig_ec.to_html(full_html=False, include_plotlyjs='cdn')
html_mun = fig_mun.to_html(full_html=False, include_plotlyjs='cdn')

# 2. Construir una página HTML limpia que contenga las tres gráficas
contenido_html = f"""
<!DOCTYPE html>
<html lang="es">
<head>
<meta charset="UTF-8">
<title>Gráficas Interactivas - Diseño de Experimentos</title>
<link href="https://fonts.googleapis.com/css2?family=Google+Sans:wght@400;500;600;700&display=swap" rel="stylesheet">
<style>
  body {{ font-family: 'Google Sans', Arial, sans-serif; background: #f7f9fc; color: #202124; margin: 0; padding: 30px; }}
  .container {{ max-width: 1200px; margin: auto; }}
  .card {{ background: white; border: 1px solid #e5e7eb; border-radius: 12px; padding: 25px; margin-bottom: 30px; box-shadow: 0 2px 4px rgba(0,0,0,0.02); }}
  h1 {{ text-align: center; color: #17365d; }}
</style>
</head>
<body>
<div class="container">
    <h1>Gráficas de Distribución (Boxplots)</h1>

    <div class="card">
        <h3>1. Escolaridad y Sexo</h3>
        {html_esc}
    </div>

    <div class="card">
        <h3>2. Estado Civil y Sexo</h3>
        {html_ec}
    </div>

    <div class="card">
        <h3>3. Municipio y Sexo</h3>
        {html_mun}
    </div>
</div>
</body>
</html>
"""

# 3. Guardar el contenido en un archivo en el entorno de Colab
nombre_archivo = "graficas_analisis.html"
with open(nombre_archivo, "w", encoding="utf-8") as f:
    f.write(contenido_html)

print(f"¡Archivo '{nombre_archivo}' generado con éxito en Colab!")

# 4. Descargar automáticamente el archivo a tu computadora
from google.colab import files
files.download(nombre_archivo)

¡Archivo 'graficas_analisis.html' generado con éxito en Colab!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>